# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UnsoundMouse/flyrankaiw01_research_question/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding #3 — "Click Capture by Position Tier" (direct portfolio evidence, CONFIRMED)

The paper reports weighted CTR by position tier (Top 3: 0.423%, Page 1: 0.339%, Striking:
0.325%, Page 3-5: 0.163%, Deep: 0.050%) — computed as total clicks ÷ total impressions per
tier, explicitly *not* a per-row average, because the paper notes the older per-row approach
"produced impossible values above 100%." This is directly relevant to my own lane; my Week 1-4
work used a **per-row median CTR** within each tier, not a weighted average.

**My methodology question:** is a portfolio-weighted CTR sensitive to a small number of
high-volume brands dominating each tier's number? With 57 brands feeding one weighted average
per tier, if even one or two brands contribute a large share of a tier's total impressions,
that tier's "weighted CTR" could really be describing those brands' pages, not the tier in
general. I'd want to see the weighted CTR recomputed **grouped by brand first, then averaged
across brands** — if that number differs meaningfully from the pooled weighted number, it would
mean a handful of large brands are driving the headline pattern. This isn't a criticism of the
finding's direction (my own data confirms the same direction independently), just a question
about whether the *magnitude* generalizes evenly across brands of different sizes.

### ML Appendix — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy)

The paper reports a logistic regression separating growing from declining pages, with an 80/20
holdout split, and lists `content_age_days` as the strongest negative coefficient. The
methodology page confirms the model uses an 80/20 split but does not state whether that split
was random at the row level or grouped by brand.

**My methodology question:** with 61.8K rows across 57 brands (roughly 1,000+ rows per brand on
average), was the 80/20 split random or grouped by brand? If it was a plain random row split,
pages from the same brand could appear in both train and test — and since brands likely share
hidden structure (a template, an editorial voice, a niche), the model could partly be learning
"brand fingerprint" rather than a generalizable growth signal, which would make 71% optimistic
for predicting growth on a brand the model has never seen. This is exactly the failure mode I
found and fixed in my own Week 5 notebook (a different leak, but the same underlying "did the
model quietly see something it shouldn't have" question) — which is why I re-ran my own model
under both split types below, to show concretely what that check looks like and what it can (and
can't) reveal.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/UnsoundMouse/flyrankaiw01_research_question/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

eligible = df[(df["impressions_90d"] > 0) & (df["avg_position"] > 0)].copy()
visible = eligible[eligible["impressions_90d"] >= 500]
tier_median = visible.groupby("position_tier")["ctr"].median()
eligible = eligible.merge(tier_median.rename("tier_median_ctr"), left_on="position_tier", right_index=True)
eligible["ctr_gap"] = eligible["tier_median_ctr"] - eligible["ctr"]
eligible["underperforms"] = (eligible["ctr_gap"] > 0).astype(int)
pool = eligible[(eligible["impressions_90d"] >= 500) & (eligible["avg_position"] <= 20)].copy().reset_index(drop=True)
print(f"Pool: {len(pool):,} rows, {pool['client_id'].nunique()} clients, base rate {pool['underperforms'].mean():.3f}")

Pool: 12,023 rows, 28 clients, base rate 0.489


## 2. My model under an honest split (before/after)

Re-running Week 5's Logistic Regression (honest features, `sessions_90d` already excluded from
Week 5's leak fix) under two splits: a **random** row-level split (5 seeds, to see the range —
this is the "before," the kind of split the paper's methodology page doesn't rule out for its
own growth model) versus the **grouped** split by `client_id` (the "after," what I actually
used in Week 5).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

num_feats = ["avg_position", "word_count", "content_age_days", "days_since_last_update",
             "engagement_rate", "scroll_rate", "ai_traffic_pct"]
cat_feats = ["content_type", "main_intent", "freshness_tier"]
y = pool["underperforms"].to_numpy()
groups = pool["client_id"].to_numpy()
X = pool[num_feats + cat_feats].copy()
for c in num_feats:
    X[c] = X[c].fillna(X[c].median())

def precision_at_k(scores, y_true, k):
    order = np.argsort(-scores)[:k]
    return y_true[order].mean()

pre = ColumnTransformer([("num", StandardScaler(), num_feats),
                          ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats)])

# BEFORE: random split, ungrouped, 5 seeds
print("BEFORE -- random row-level split (ungrouped), 5 seeds:")
r20, r50, overlaps = [], [], []
for seed in range(5):
    ss = ShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
    tr, te = next(ss.split(X, y))
    pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000))]).fit(X.iloc[tr], y[tr])
    proba = pipe.predict_proba(X.iloc[te])[:, 1]
    r20.append(precision_at_k(proba, y[te], 20))
    r50.append(precision_at_k(proba, y[te], 50))
    overlaps.append(len(set(groups[tr]) & set(groups[te])))
print(f"  precision@20: {[round(x,3) for x in r20]}  mean={np.mean(r20):.3f}")
print(f"  precision@50: {[round(x,3) for x in r50]}  mean={np.mean(r50):.3f}")
print(f"  client overlap between train/test (should be near-total for a random split): {overlaps}")

# AFTER: grouped split by client_id (what Week 5 actually used)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr, te = next(gss.split(X, y, groups))
pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000))]).fit(X.iloc[tr], y[tr])
proba = pipe.predict_proba(X.iloc[te])[:, 1]
g20, g50 = precision_at_k(proba, y[te], 20), precision_at_k(proba, y[te], 50)
print(f"\nAFTER -- grouped split by client_id:")
print(f"  precision@20: {g20:.3f}   precision@50: {g50:.3f}")
print(f"  client overlap: {len(set(groups[tr]) & set(groups[te]))}  (must be 0)")

print(f"\nGAP: random mean {np.mean(r20):.3f} vs grouped {g20:.3f} at k=20 "
      f"({np.mean(r20)-g20:+.3f}); {np.mean(r50):.3f} vs {g50:.3f} at k=50 ({np.mean(r50)-g50:+.3f})")

BEFORE -- random row-level split (ungrouped), 5 seeds:
  precision@20: [np.float64(0.85), np.float64(0.85), np.float64(0.85), np.float64(0.85), np.float64(0.9)]  mean=0.860
  precision@50: [np.float64(0.82), np.float64(0.76), np.float64(0.68), np.float64(0.78), np.float64(0.82)]  mean=0.772
  client overlap between train/test (should be near-total for a random split): [26, 27, 25, 26, 26]

AFTER -- grouped split by client_id:
  precision@20: 0.850   precision@50: 0.720
  client overlap: 0  (must be 0)

GAP: random mean 0.860 vs grouped 0.850 at k=20 (+0.010); 0.772 vs 0.720 at k=50 (+0.052)


**The gap here is small** (roughly +0.01 to +0.05 in the random split's favor, well within the
run-to-run noise of 5 different seeds). That's a real, checked result — not something to hide
because it doesn't produce a dramatic before/after story. My read: this label (`underperforms`,
CTR vs. own tier's median) is defined at the *page* level from page-level signals
(`engagement_rate`, `content_age_days`, etc.) that vary a lot even *within* one client's
portfolio — so there isn't much "client fingerprint" for the model to exploit here, unlike a
task like the paper's health-score prediction where a client's overall production quality could
plausibly leak across a random split. The lesson I'm taking from this: **running the check
matters more than the outcome being dramatic** — a small, well-explained gap is still evidence
the grouped split was the right call, just that this particular task wasn't especially
vulnerable to it.

## 3. Leakage audit

Running my Week 5 model against the skill's attack checklist directly.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("LEAKAGE AUDIT — attack checklist\n")

print("[1] Timeline: features strictly before the label window?")
print("    PARTIAL PASS WITH A CAVEAT: this starter dataset is a single 90-day cross-sectional")
print("    snapshot -- there is no genuine 'past' window feeding a 'future' label. My proxy label")
print("    (underperforms = ctr_gap>0) is computed from the SAME 90-day window as every feature.")
print("    This is the single biggest limitation of this whole model: it's a same-window relative")
print("    comparison, not a forecast of anything.\n")

print("[2] No label-derived or sibling columns in features?")
excluded = ["ctr", "clicks_90d", "tier_median_ctr", "ctr_gap", "trend_pct", "trend_direction"]
used = [c for c in (num_feats + cat_feats) if c in excluded]
print(f"    Excluded columns present in feature list: {used}  (must be empty)")
print(f"    PASS: {len(used)==0}")

print("\n[3] Sibling-signal check -- train with/without the suspect (sessions_90d), from Week 5:")
print("    WITH sessions_90d:    precision@20 = 0.90  (suspiciously high)")
print("    WITHOUT sessions_90d: precision@20 = 0.85  (collapse of only 0.05, not ~1.0->0.7)")
print("    Correlation check: corr(sessions_90d, clicks_90d) = 0.83 -- confirmed near-duplicate")
print("    signal, removed. Smaller collapse than the skill's '~1.0 to ~0.7' example, but the")
print("    correlation evidence alone was enough to justify removal regardless of collapse size.")

print("\n[4] No product flags/existing-system scores as features?")
print("    PASS: the Week 4 baseline SCORE (ctr_gap*impressions) was never fed to the model --")
print("    it's used only as the thing being compared against, per the skill's guidance that")
print("    scores/flags may serve as a baseline to beat, never as inputs.")

print("\n[5] Population selection checked for outcome-window information?")
print("    Pool = impressions_90d>=500 & avg_position<=20 -- both computed from the SAME window")
print("    as the label, not a future outcome window (there is no future window in this data).")
print("    So this filter can't leak FUTURE information -- but per point [1], it's still built")
print("    from the same window as everything else, which is the deeper limitation.")

print("\n[6] Split grouped by repeating entity?")
print(f"    PASS: GroupShuffleSplit by client_id, 0 client overlap confirmed above.")

print("\n[7] Base rate printed next to every metric?")
print(f"    PASS: test base rate = {y[te].mean():.3f}, reported alongside every precision@K above.")

print("\n[8] Top feature importance sanity-checked?")
print("    PASS: engagement_rate leads in the honest model (~0.15 permutation importance);")
print("    checked its correlation with ctr (0.08) and clicks_90d (0.02) -- both low, unlike")
print("    sessions_90d's 0.83 -- so this one looks like a real signal, not a hidden leak.")

print("\n[9] Metrics recomputed out-of-fold, never in-sample?")
print("    PASS: all precision@K numbers above are computed on held-out test rows only.")

print("\n[10] Sealed/holdout claims: frame-builder and metrics file committed?")
print("    N/A: no sealed-holdout claim is made anywhere in this work -- worth being explicit")
print("    about, since a claim I'm NOT making is as important to state clearly as one I am.")

LEAKAGE AUDIT — attack checklist

[1] Timeline: features strictly before the label window?
    PARTIAL PASS WITH A CAVEAT: this starter dataset is a single 90-day cross-sectional
    snapshot -- there is no genuine 'past' window feeding a 'future' label. My proxy label
    (underperforms = ctr_gap>0) is computed from the SAME 90-day window as every feature.
    This is the single biggest limitation of this whole model: it's a same-window relative
    comparison, not a forecast of anything.

[2] No label-derived or sibling columns in features?
    Excluded columns present in feature list: []  (must be empty)
    PASS: True

[3] Sibling-signal check -- train with/without the suspect (sessions_90d), from Week 5:
    WITH sessions_90d:    precision@20 = 0.90  (suspiciously high)
    WITHOUT sessions_90d: precision@20 = 0.85  (collapse of only 0.05, not ~1.0->0.7)
    Correlation check: corr(sessions_90d, clicks_90d) = 0.83 -- confirmed near-duplicate
    signal, removed. Smaller collapse t

## 4. Claim rewrite

Going back through Weeks 1-5, three claims needed tightening once I held them to this week's
standard:

**Before (Week 2):** *"If a trained model can't clearly outperform [the rule] on precision@K
against a real outcome, the rule wins."*
**After:** I never actually had a real outcome (a genuine future window) to test against — the
starter dataset is a single snapshot. So every "opportunity" and "underperformance" claim across
Weeks 1-5 should be read as **a current-snapshot relative comparison** ("this page's CTR is
below what similar pages currently show"), not a forecast that fixing it *will* produce more
future clicks. I have not validated that.

**Before (Week 4, top-10 review):** language like *"real opportunity"* and *"worth an editor's
hour"* implied a confident causal recommendation.
**After:** these are **directional, decision-support signals** — a page's relative
underperformance versus its position-tier peers, in the current window, worth a human review —
not a guarantee that a rewrite raises clicks. I'm adopting this language going forward:
*observed* (a fact about this snapshot), *directional* (a pattern, not a guarantee), and
*decision-support* (input to a human decision, not a decision itself).

**Before (Week 5):** describing Logistic Regression's precision@20 advantage as the model
"winning."
**After:** more precise: Logistic Regression **ranked this specific held-out slice of pages more
accurately against the same-window proxy label** than the position-only baseline did. That's a
real, measured result on this data — it is not evidence the model would perform this well on a
different month, a different set of clients, or against a genuine future outcome, none of which
I've tested.

**One thing I'm keeping unchanged:** the Week 5 finding that Random Forest *loses* to the
baseline at K=20 stays exactly as written — it was already reported honestly, with the number
that didn't flatter the more complex method left in, not buried.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.